## Image Input

In [1]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png',multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [2]:
print(uploader.value)

({'name': 'Moon_Capital.png', 'type': 'image/png', 'size': 10536882, 'content': <memory at 0x00000289E6A57280>, 'last_modified': datetime.datetime(2026, 6, 29, 20, 43, 10, 808000, tzinfo=datetime.timezone.utc)},)


In [2]:
import base64
# Get the first uploaded file
uploaded_file = uploader.value[0]
# This is memory view of the uploaded file
content_mv = uploaded_file['content']
# Convert memory view to bytes
img_bytes = bytes(content_mv)

img_64 = base64.b64encode(img_bytes).decode('utf-8')

IndexError: tuple index out of range

In [19]:
from langchain.messages import HumanMessage
multimodal_question = HumanMessage(content=[
  {"type":"text","text":"What is in this image?"},
  {"type":"image","base64":img_64,"mime_type":"image/png"}
])


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="gemini-3.1-flash-lite",temperature=0.2,model_provider="google_genai")


In [22]:
from langchain.agents import create_agent
agent = create_agent(model=model,system_prompt="You are a science fiction writer.")

In [23]:
response = agent.invoke({"messages":[multimodal_question]})
print(response['messages'][-1].content)

[{'type': 'text', 'text': 'This image depicts **Moon City Capital**, a sprawling, futuristic lunar metropolis established in the year 2068. As a science fiction writer, I see this as a testament to humanity’s transition into a multi-planetary species.\n\nHere is a breakdown of the scene:\n\n*   **The Architecture:** The city is built within a lunar crater, utilizing a series of interconnected geodesic domes to maintain a pressurized, breathable atmosphere for its inhabitants. At the center stands the "Luna Dome," the heart of the city, surrounded by the towering "Administration Tower."\n*   **The Infrastructure:** A sophisticated network of elevated, transparent transit tubes connects the various sectors of the city, allowing for efficient movement across the rugged lunar landscape.\n*   **The Gateway:** In the foreground, we see the "Moon City Port," a bustling spaceport where sleek, shuttle-like spacecraft are docked, ready for transit between the Moon and Earth.\n*   **The Setting:*

## Audio input

In [9]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

#Recording settings
duration = 5  # seconds
sample_rate = 44100  # Hz

print("Recording...")

audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)

#Progress bar for recording
for _ in tqdm(range(duration*10)):
    time.sleep(0.1)
sd.wait()
print("Done.")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode('utf-8')


Recording...


100%|██████████| 50/50 [00:05<00:00,  9.73it/s]

Done.


In [11]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
agent = create_agent(model=model)

multimodal_question = HumanMessage(content=[
  {"type":"text","text":"Tell me about this audio file"},
  {"type":"audio","base64":aud_b64,"mime_type":"audio/wav"}
])

response = agent.invoke({"messages":[multimodal_question]})
print(response['messages'][-1].content[0]["text"]) 

What is the coolest language in the world? It is mostly used
